
# Gold Layer Table: `gold.top_tags`

## Overview

The `gold.top_tags` table provides a business-ready aggregation of the most frequently used tags across all posts.

This dataset represents a **high-level summary of tagging behavior across posts**.

---

## Input Schema

### `silver.posts`

| Column    | Type          | Description                         |
| --------- | ------------- | ----------------------------------- |
| PostId    | BIGINT        | Unique identifier of a post         |
| TagsArray | ARRAY<STRING> | List of tags associated with a post |

---

## Transformation Logic

The following transformations are applied:

1. Filter out records where `TagsArray` is NULL
2. Explode `TagsArray` into individual tag rows
3. Group by tag
4. Count distinct posts per tag
5. Sort by popularity (descending)
6. Limit results to top 100 tags

---

## Output Schema

### `gold.top_tags`

| Column     | Type   | Description                                 |
| ---------- | ------ | ------------------------------------------- |
| tag        | STRING | Tag name                                    |
| post_count | BIGINT | Number of distinct posts containing the tag |

---

## Business Meaning

Each row represents:

> “How many unique posts use this tag?”

This allows stakeholders to:

* Identify trending topics
* Understand content distribution
* Track platform usage patterns

---

## Design Decisions

### Exact Counting

Uses `COUNT(DISTINCT PostId)` to ensure accurate metrics.

### Flattened Structure

Tags are exploded into a normalized format for aggregation.

### Limited Output

Only top 100 tags are stored to optimize performance and usability.

### Gold Layer Ready

This dataset is fully curated for direct consumption by analytics tools.

---

## Notes & Assumptions

* Assumes `PostId` is unique per post
* Posts with NULL `TagsArray` are excluded
* Tag casing and normalization are assumed to be handled upstream in the Silver layer

---

## Refresh Strategy

* Current approach: **full table overwrite**
* Suitable for batch processing pipelines

For larger scale systems, consider:

* Incremental updates
* Partitioned aggregation
* Materialized views

---

## Example Use Cases

* “Top programming languages this month”
* “Most popular discussion topics”
* Dashboard KPI: tag distribution trends

---

## Summary

`gold.top_tags` is a curated analytical dataset that transforms raw tag arrays into a ranked business metric showing tag popularity across posts.


In [0]:
%sql
CREATE OR REPLACE TABLE `data-plataform-jayzern`.default.marts_top_tags AS
WITH exploded_tags AS (
    SELECT
        PostId,
        tag
    FROM `data-plataform-jayzern`.default.stg_posts
    LATERAL VIEW explode(TagsArray) t AS tag
    WHERE TagsArray IS NOT NULL
)

SELECT
    tag,
    COUNT(DISTINCT PostId) AS post_count
FROM exploded_tags
GROUP BY tag
ORDER BY post_count DESC
LIMIT 10;

In [0]:
%sql
SELECT *
FROM `data-plataform-jayzern`.default.marts_top_tags
LIMIT 10;